In [1]:
import jax
from jax import vmap
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

from DGPs import DGP_univariate, DGP_high_dim_X, DGP_high_dim_M, DGP_high_dim_MX, DGP_high_dim_Y, DGP_binary

from propensities import estimate_propensity
from utils import Array,Scalar
from KRR_utils import learn_hyperparams
from experiments import run_convergence_experiment

In [ ]:
dim=1
treatment=1
method='table'
A_truth,M_truth,X_truth,Y_truth = DGP_binary(10000,seed=101,dim=dim,gt=True,a0=treatment)

A,M,X,Y = DGP_binary(2000,seed=0,dim=dim)

params=learn_hyperparams(A,M,X,Y,subsample=False)

In [ ]:
n_list=[250,500,1000,2000,4000,8000]
num_seeds=200
truth_table=[True,False]
dr_mat=jnp.zeros((len(n_list),2,num_seeds))
pi_mat=jnp.zeros((len(n_list),2,num_seeds))
for i in range(len(n_list)):
    n=n_list[i]
    print('n: ',n)
    for j in range(2):
        explicit_y=truth_table[j]
        for k in range(num_seeds):
            if k%50 == 0:
                print('seed: ',k)
            dr,pi=run_convergence_experiment(DGP_binary, n, 
                                             seed=k, 
                                             params=params, 
                                             true_data=(A_truth,M_truth,X_truth,Y_truth), 
                                             treatment=treatment,
                                             x_method=method,
                                             mx_method=method,
                                             m=True,
                                             y=explicit_y)
            dr_mat = dr_mat.at[i,j,k].set(dr)
            pi_mat = pi_mat.at[i,j,k].set(pi)

with open('results/binary_tabular_v2.npy', 'wb') as f:
    jnp.save(f, dr_mat)
    jnp.save(f, pi_mat)